In [3]:
# Importing
import pandas as pd
import numpy as np
from utils.fake_na_detection_and_cleaning import detect_fake_nulls,replace_fake_nulls
from utils.sql_connector import connect_sql,read_sql_query,write_to_sql
from utils.data_type_converter import string_to_category,string_to_numeric
from utils.normalizer import normalize_categorical_columns_exploding,normalize_categorical_columns_manual_mapping,normalize_na_replacer_columns,normalize_categorical_columns_non_exploding,clean_years_columns,fill_na_and_remove_outlier_percentaile_method


In [4]:
# loading of 2021 datasets
query='select * from Bronze.Survey_2021'
database='Stack_Overflow_Survey'
conn, engine = connect_sql('Stack_Overflow_Survey')
Survey_2021_df = read_sql_query(query, conn)

Successfully Connected


c:\Users\Ayush\Git Repo\Stack-Over-Flow-Survey-Data-Engineering-Project\Data Warehouse\Silver Layer\utils\sql_connector.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


Query executed successfully


In [18]:
Survey_2021_df_cleaned=Survey_2021_df.copy()
detect_fake_nulls(Survey_2021_df_cleaned)
replace_fake_nulls(Survey_2021_df_cleaned)

categorical_cols = [
    "MainBranch", "Employment", "EdLevel", "Age", "Age1stCode",
    "LearnCode", "OpSys", "OrgSize", "Country", "US_State", "UK_Country",
    "Gender", "Trans", "Sexuality", "Ethnicity", "Accessibility",
    "MentalHealth", "SOVisitFreq", "SOAccount", "SOPartFreq", "SOComm",
    "NEWSOSites", "NEWOtherComms", "NEWStuck", "CompFreq",
    "SurveyLength", "SurveyEase", "DevType"
]



{'Accessibility': {'NA': np.int64(5836)},
 'Age': {'NA': np.int64(1032)},
 'Age1stCode': {'NA': np.int64(196)},
 'CompFreq': {'NA': np.int64(31289)},
 'CompTotal': {'NA': np.int64(36256)},
 'ConvertedCompYearly': {'NA': np.int64(36595)},
 'Currency': {'NA': np.int64(22359)},
 'DatabaseHaveWorkedWith': {'NA': np.int64(13893)},
 'DatabaseWantToWorkWith': {'NA': np.int64(25140)},
 'DevType': {'NA': np.int64(16955)},
 'EdLevel': {'NA': np.int64(313)},
 'Employment': {'NA': np.int64(116)},
 'Ethnicity': {'NA': np.int64(3975)},
 'Gender': {'NA': np.int64(1153)},
 'LanguageHaveWorkedWith': {'NA': np.int64(1082)},
 'LanguageWantToWorkWith': {'NA': np.int64(6618)},
 'LearnCode': {'NA': np.int64(476)},
 'MentalHealth': {'NA': np.int64(6519)},
 'MiscTechHaveWorkedWith': {'NA': np.int64(36384)},
 'MiscTechWantToWorkWith': {'NA': np.int64(45418)},
 'NEWCollabToolsHaveWorkedWith': {'NA': np.int64(2205)},
 'NEWCollabToolsWantToWorkWith': {'NA': np.int64(10417)},
 'NEWOtherComms': {'NA': np.int64(611)

In [19]:
# Mormalization of categorical columns : Basic mapping of values to reduce the number of unique values in each column and make it more consistent for analysis and visualization.

employment_map = {
    'Employed full-time': 'Employed',
    'Employed part-time': 'Employed',
    'Independent contractor, freelancer, or self-employed': 'Freelance',
    'Student, full-time': 'Student',
    'Student, part-time': 'Student',
    'Not employed, but looking for work': 'Unemployed',
    'Not employed, and not looking for work': 'Unemployed',
    'Retired': 'Retired',
    'I prefer not to say': 'I prefer not to say',
    'Nan' : 'Not Available',
}
ed_level_map = {
    'BachelorΓÇÖs degree (B.A., B.S., B.Eng., etc.)': 'Undergraduate',
    'MasterΓÇÖs degree (M.A., M.S., M.Eng., MBA, etc.)': 'Postgraduate',
    'Some college/university study without earning a degree': 'Undergraduate',
    'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)': 'High School',
    'Other doctoral degree (Ph.D., Ed.D., etc.)': 'Doctorate',
    'Primary/elementary school': 'Primary',
    'Associate degree (A.A., A.S., etc.)': 'Undergraduate',
    'Something else': 'Other',
    'Professional degree (JD, MD, etc.)': 'Postgraduate'
}
age_1st_code_map = {
    'Younger than 5 years': '<5',
    '5 - 10 years': '5-10',
    '11 - 17 years': '11-17',
    '18 - 24 years': '18-24',
    '25 - 34 years': '25-34',
    '35 - 44 years': '35-44',
    '45 - 54 years': '45-54',
    '55 - 64 years': '55-64',
    'Older than 64 years': '>64'
}
age_map = {
    'Under 18 years old': 'less than 18',
    '18-24 years old': '18-24',
    '25-34 years old': '25-34',
    '35-44 years old': '35-44',
    '45-54 years old': '45-54',
    '55-64 years old': '55-64',
    '65 years or older': 'greater than65',
    'Prefer not to say': 'Unknown'
}
operating_system_map = {
    'Windows Subsystem for Linux (WSL)': 'WSL',
    'Linux-based': 'Linux',
    'MacOS': 'MacOS',
    'Windows': 'Windows',
    'BSD': 'Unix/BSD',
    'Other (please specify):': 'Other'
}
org_mapping = {
    'Just me - I am a freelancer, sole proprietor, etc.': 'Self-employed',
    '2 to 9 employees': 'Micro (2-9)',
    '10 to 19 employees': 'Small (10-19)',
    '20 to 99 employees': 'Small-Medium (20-99)',
    '100 to 499 employees': 'Medium (100-499)',
    '500 to 999 employees': 'Large (500-999)',
    '1,000 to 4,999 employees': 'Enterprise (1K-4.9K)',
    '5,000 to 9,999 employees': 'Enterprise (5K-9.9K)',
    '10,000 or more employees': 'Enterprise (10K+)'
}

trans_map = {
    'No': 'No',
    'Yes': 'Yes',
    'Prefer not to say': 'Unknown',
    'Or, in your own words:': 'Self-described'
}
visit_freq_map = {
    'Multiple times per day': 'High (Multiple/Day)',
    'Daily or almost daily': 'High (Daily)',
    'A few times per week': 'Medium (Weekly)',
    'A few times per month or weekly': 'Medium (Monthly/Weekly)',
    'Less than once per month or monthly': 'Low (Monthly)',
}
so_account_map = {
    'Yes': 'Yes',
    'No': 'No',
    '''Not sure/can't remember''': 'Uncertain',
}
part_freq_map = {
    'Multiple times per day': 'High (Multiple/Day)',
    'Daily or almost daily': 'High (Daily)',
    'A few times per week': 'Medium (Weekly)',
    'A few times per month or weekly': 'Medium (Monthly/Weekly)',
    'Less than once per month or monthly': 'Low (Monthly)',
    'I have never participated in Q&A on Stack Overflow': 'Never'
}
comm_map = {
    'Yes, definitely': 'Strongly Positive',
    'Yes, somewhat': 'Positive',
    'Neutral': 'Neutral',
    'No, not really': 'Negative',
    'No, not at all': 'Strongly Negative',
    'Not sure': 'Uncertain'
}
new_so_sites_map = {
    'Stack Overflow;Stack Exchange': 'SO & Stack Exchange',
    'Stack Overflow': 'Stack Overflow Only',
    'Stack Overflow;Stack Exchange;Stack Overflow for Teams (private knowledge sharing & collaboration platform for companies)': 'Full Network User',
    'I have never visited Stack Overflow or the Stack Exchange network': 'Never Visited',
    'Stack Overflow;Stack Overflow for Teams (private knowledge sharing & collaboration platform for companies)': 'SO & Teams',
    'Stack Exchange': 'Stack Exchange Only',
    'Stack Overflow for Teams (private knowledge sharing & collaboration platform for companies)': 'Teams Only',
    'Stack Exchange;Stack Overflow for Teams (private knowledge sharing & collaboration platform for companies)': 'SE & Teams'
}
survey_length_map = {
    'Appropriate in length': 'Appropriate',
    'Too long': 'Too Long',
    'Too short': 'Too Short'
}
survey_ease_map = {
    'Easy': 'Easy',
    'Neither easy nor difficult': 'Neutral',
    'Difficult': 'Difficult'
}
un_normalized_cols_name = [
    'Employment', 'EdLevel', 'Age', 'Age1stCode', 'OpSys', 'OrgSize', 'Trans', 'SOVisitFreq', 'SOAccount', 'SOPartFreq', 'SOComm', 'NEWSOSites', 'SurveyLength', 'SurveyEase'
]
normalized_cols_name=[
    'Employment','Education_Level','Age','AgeCode','OperatingSystem','Organization_Size',"TransGender","StackOverflow_Visit_Frequency","StackOverflow_Account_exists","StackOverflow_Participation_Frequency","StackOverflow_Community_Experience","NewStackOverflow_Sites",'Survey_Length','Survey_Ease'
]
columns_map=[
    employment_map, ed_level_map, age_map, age_1st_code_map, operating_system_map,org_mapping, trans_map, visit_freq_map, so_account_map, part_freq_map, comm_map, new_so_sites_map, survey_length_map, survey_ease_map
]

Survey_2021_df_cleaned = normalize_categorical_columns_manual_mapping(Survey_2021_df_cleaned, un_normalized_cols_name, normalized_cols_name, columns_map)


['Independent contractor, freelancer, or self-employed'
 'Student, full-time' 'Employed full-time' 'Student, part-time'
 'I prefer not to say' 'Employed part-time'
 'Not employed, but looking for work' 'Retired'
 'Not employed, and not looking for work' nan]
Employment
Employed                       56045
Student                        13832
Freelance                       8041
Unemployed                      4189
I prefer not to say              890
Retired                          326
Not Available or Applicable      116
Name: count, dtype: int64
['Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)'
 'BachelorΓÇÖs degree (B.A., B.S., B.Eng., etc.)'
 'MasterΓÇÖs degree (M.A., M.S., M.Eng., MBA, etc.)'
 'Other doctoral degree (Ph.D., Ed.D., etc.)'
 'Some college/university study without earning a degree' 'Something else'
 'Professional degree (JD, MD, etc.)' 'Primary/elementary school'
 'Associate degree (A.A., A.S., etc.)' nan]
Education_Level
Undergrad

In [20]:
# Mormalization of categorical columns : changing NA to more meaningful values and make it more consistent for analysis and visualization.

nan_replacer_columns=[
    'US_State', 'UK_Country', 'NEWOtherComms','CompFreq'
    ]
cleaned_nan_replacer_columns=[
    'USA_State', 'UK_Country', 'Know_Other_Community','Compensation_Frequency'
]
Survey_2021_df_cleaned=normalize_na_replacer_columns(Survey_2021_df_cleaned,nan_replacer_columns, cleaned_nan_replacer_columns)

In [21]:
# Mormalization of categorical columns : Multi-select columns where respondents could select multiple options, resulting in semicolon-separated values. We will map the individual options to broader categories and also create a new category for respondents who selected multiple options.

gender_map = {
    'Man': 'Man',
    'Woman': 'Woman',
    'Non-binary, genderqueer, or gender non-conforming': 'Non-binary / GNC',
    'Prefer not to say': 'Unknown',
    'Or, in your own words:': 'Non-binary / GNC',
}
ethnicity_map = {
    'White or of European descent': 'White',
    'South Asian': 'South Asian',
    'Middle Eastern': 'Middle Eastern',
    'Southeast Asian': 'Southeast Asian',
    'East Asian': 'East Asian',
    'Hispanic or Latino/a/x': 'Hispanic/Latino',
    'Black or of African descent': 'Black',
    'Indigenous (such as Native American, Pacific Islander, or Indigenous Australian)': 'Indigenous',
    'Multiracial': 'Multiracial/Biracial',
    'Biracial': 'Multiracial/Biracial',
    'Or, in your own words:': 'Self-described',
    'Prefer not to say': 'Unknown',
    "I don't know": 'Unknown'
}
learn_code_map = {
    'School': 'Academic (Degree)',
    'Coding Bootcamp': 'Bootcamp',
    'Online Courses or Certification': 'Online Certifications',
    'Other online resources (ex: videos, blogs, etc)': 'Digital Self-Taught',
    'Books / Physical media': 'Physical Media',
    'Online Forum': 'Community/Forums',
    'Colleague': 'Professional Network',
    'Friend or family member': 'Social/Family',
    'Other (please specify):': 'Other'
}
sexuality_map = {
    'Straight / Heterosexual': 'Straight',
    'Bisexual': 'Bisexual',
    'Gay or Lesbian': 'Gay or Lesbian',
    'Queer': 'Queer',
    'Prefer to self-describe:': 'Self-described',
    'Prefer not to say': 'Unknown'
}
accessibility_map = {
    'None of the above': 'None',
    'I am deaf / hard of hearing': 'Hearing Impairment',
    'I am blind / have difficulty seeing': 'Visual Impairment',
    'I am unable to / find it difficult to type': 'Mobility (Typing)',
    'I am unable to / find it difficult to walk or stand without assistance': 'Mobility (Walking/Standing)',
    'Or, in your own words:': 'Self-described',
    'Prefer not to say': 'Unknown'
}
mental_health_map = {
    'None of the above': 'None',
    'I have a concentration and/or memory disorder (e.g. ADHD)': 'ADHD/Concentration',
    'I have an anxiety disorder': 'Anxiety',
    'I have a mood or emotional disorder (e.g. depression, bipolar disorder)': 'Mood/Emotional',
    "I have autism / an autism spectrum disorder (e.g. Asperger's)": 'Autism Spectrum',
    'Or, in your own words:': 'Self-described',
    'Prefer not to say': 'Unknown'
}

multi_select_cols = [
    "Gender", "Sexuality", "Ethnicity", 
    "Accessibility", "MentalHealth"
]
multi_select_normalized = [
     "Gender_Clean", "Sexuality_Clean", "Ethnicity_Clean", 
    "Accessibility_Status", "Mental_Health_Status"
]
multi_select_maps = [
    gender_map,       
    sexuality_map,     
    ethnicity_map,     
    accessibility_map, 
    mental_health_map
]
normalize_categorical_columns_non_exploding(Survey_2021_df_cleaned, multi_select_cols, multi_select_normalized, multi_select_maps)

Gender_Clean
Man                   74817
Woman                  4120
Unknown                2595
Non-binary / GNC       1103
Diverse / Multiple      804
Name: count, dtype: int64
Sexuality_Clean
Straight              61094
Unknown               14856
Bisexual               2879
Diverse / Multiple     1609
Gay or Lesbian         1367
Self-described         1258
Queer                   376
Name: count, dtype: int64
Ethnicity_Clean
White                   42671
Unknown                  9177
South Asian              8328
Diverse / Multiple       5690
Hispanic/Latino          3585
Southeast Asian          3224
Middle Eastern           2985
East Asian               2947
Black                    2085
Self-described           2014
Multiracial/Biracial      650
Indigenous                 83
Name: count, dtype: int64
Accessibility_Status
None                           72725
Unknown                         7754
Visual Impairment               1030
Self-described                   842
Hearing Impa

,ResponseId,MainBranch,Employment,Country,US_State,UK_Country,EdLevel,Age1stCode,LearnCode,YearsCode,...,Survey_Length,Survey_Ease,USA_State,Know_Other_Community,Compensation_Frequency,Gender_Clean,Sexuality_Clean,Ethnicity_Clean,Accessibility_Status,Mental_Health_Status
0,1,I am a developer by profession,Freelance,Slovakia,NaN,Not Available or Applicable,"Secondary school (e.g. American high school, G...",18 - 24 years,Coding Bootcamp;Other online resources (ex: vi...,NaN,...,Appropriate,Easy,Not Available or Applicable,No,Monthly,Man,Straight,White,None,None
1,2,I am a student who is learning to code,Student,Netherlands,NaN,Not Available or Applicable,"BachelorΓÇÖs degree (B.A., B.S., B.Eng., etc.)",11 - 17 years,"Other online resources (ex: videos, blogs, etc...",7,...,Appropriate,Easy,Not Available or Applicable,No,Not Available or Applicable,Man,Straight,White,None,None
2,3,"I am not primarily a developer, but I write co...",Student,Russian Federation,NaN,Not Available or Applicable,"BachelorΓÇÖs degree (B.A., B.S., B.Eng., etc.)",11 - 17 years,"Other online resources (ex: videos, blogs, etc...",NaN,...,Appropriate,Easy,Not Available or Applicable,Yes,Not Available or Applicable,Man,Unknown,Unknown,None,None
3,4,I am a developer by profession,Employed,Austria,NaN,Not Available or Applicable,"MasterΓÇÖs degree (M.A., M.S., M.Eng., MBA, etc.)",11 - 17 years,NaN,NaN,...,Appropriate,Neutral,Not Available or Applicable,No,Monthly,Man,Straight,White,Hearing Impairment,Unknown
4,5,I am a developer by profession,Freelance,United Kingdom of Great Britain and Northern I...,NaN,England,"MasterΓÇÖs degree (M.A., M.S., M.Eng., MBA, etc.)",5 - 10 years,Friend or family member,17,...,Appropriate,Easy,Not Available or Applicable,No,Not Available or Applicable,Man,Unknown,White,None,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83434,83435,I am a developer by profession,Employed,United States of America,Texas,Not Available or Applicable,"BachelorΓÇÖs degree (B.A., B.S., B.Eng., etc.)",11 - 17 years,"Other online resources (ex: videos, blogs, etc...",6,...,Appropriate,Easy,Texas,No,Yearly,Man,Straight,White,None,ADHD/Concentration
83435,83436,I am a developer by profession,Freelance,Benin,NaN,Not Available or Applicable,"BachelorΓÇÖs degree (B.A., B.S., B.Eng., etc.)",11 - 17 years,"Other online resources (ex: videos, blogs, etc...",4,...,Appropriate,Easy,Not Available or Applicable,No,Monthly,Man,Straight,Black,None,None
83436,83437,I am a developer by profession,Employed,United States of America,New Jersey,Not Available or Applicable,"Secondary school (e.g. American high school, G...",11 - 17 years,School,10,...,Appropriate,Neutral,New Jersey,No,Weekly,Man,Unknown,White,None,None
83437,83438,I am a developer by profession,Employed,Canada,NaN,Not Available or Applicable,"BachelorΓÇÖs degree (B.A., B.S., B.Eng., etc.)",11 - 17 years,Online Courses or Certification;Books / Physic...,5,...,Appropriate,Neutral,Not Available or Applicable,No,Monthly,Man,Straight,White,None,Mood/Emotional


In [22]:
# Mormalization of categorical columns :  we will create a mapping to group similar roles and responses together, reducing the number of unique values while preserving the overall meaning.

dev_type_map = {
    'Developer, full-stack': 'Full-stack', 'Developer, back-end': 'Back-end',
    'Developer, front-end': 'Front-end', 'Developer, mobile': 'Mobile',
    'Developer, desktop or enterprise applications': 'Desktop/Enterprise',
    'Engineer, data': 'Data Engineer', 'Data scientist or machine learning specialist': 'Data Scientist/ML',
    'Data or business analyst': 'Data/BI Analyst', 'DevOps specialist': 'DevOps',
    'Engineer, site reliability': 'SRE', 'Engineering manager': 'Engineering Manager',
    'Senior Executive (C-Suite, VP, etc.)': 'Executive', 'System administrator': 'SysAdmin',
    'Database administrator': 'DBA', 'Developer, game or graphics': 'Game/Graphics',
    'Developer, embedded applications or devices': 'Embedded/IoT', 'Developer, QA or test': 'QA/Testing',
    'Academic researcher': 'Researcher', 'Scientist': 'Scientist', 'Student': 'Student',
    'Educator': 'Educator', 'Designer': 'Designer', 'Product manager': 'Product Manager',
    'Marketing or sales professional': 'Marketing/Sales', 'Other (please specify):': 'Other'
}

stuck_map = {
    'Google it': 'Search (Google)', 'Visit Stack Overflow': 'Community (Stack Overflow)',
    'Visit another developer community (please name):': 'Community (Other)',
    'Watch help / tutorial videos': 'Self-Paced Learning', 'Go for a walk or other physical activity': 'Physical Break',
    'Meditate': 'Mental Break', 'Play games': 'Mental Break', 'Do other work and come back later': 'Context Switch',
    'Call a coworker or friend': 'Social Support', 'Panic': 'Emotional Response', 'Other (please specify):': 'Other'
}
learn_code_map = {
    'School': 'Academic (Degree)',
    'Coding Bootcamp': 'Bootcamp',
    'Online Courses or Certification': 'Online Certifications',
    'Other online resources (ex: videos, blogs, etc)': 'Digital Self-Taught',
    'Books / Physical media': 'Physical Media',
    'Online Forum': 'Community/Forums',
    'Colleague': 'Professional Network',
    'Friend or family member': 'Social/Family',
    'Other (please specify):': 'Other'
}
tech_stack_cols = [
    'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 
    'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 
    'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 
    'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 
    'MiscTechHaveWorkedWith', 'MiscTechWantToWorkWith', 
    'ToolsTechHaveWorkedWith', 'ToolsTechWantToWorkWith', 
    'NEWCollabToolsHaveWorkedWith', 'NEWCollabToolsWantToWorkWith'
]
manual_mapping_cols = [
    'DevType', 'NEWStuck'
]
all_target_cols = manual_mapping_cols + tech_stack_cols
manual_maps = {
    'DevType': dev_type_map,
    'NEWStuck': stuck_map,
    'LearnCode':learn_code_map
}

bridge_results = {}
target_names = [
    col + '_Clean' 
    for col in all_target_cols
]
maps_to_use = [
    manual_maps.get(col) 
    for col in all_target_cols
]


bridge_results = normalize_categorical_columns_exploding(
    Survey_2021_df_cleaned, 
    all_target_cols, 
    target_names, 
    maps_to_use
)

--- Distribution for DevType_Clean ---
DevType_Clean
Full-stack             32891
Back-end               29071
Front-end              18231
Other/Unknown          16955
Desktop/Enterprise     11036
Mobile                  9800
DevOps                  7058
SysAdmin                6079
DBA                     5655
Designer                4611
Embedded/IoT            4598
Data Scientist/ML       4273
Student                 4187
Data Engineer           4176
Engineering Manager     3810
Data/BI Analyst         3792
QA/Testing              3611
Other                   3545
Product Manager         3074
Researcher              2899
SRE                     2448
Educator                2369
Game/Graphics           2112
Executive               2103
Scientist               2015
Marketing/Sales          638
Name: count, dtype: int64
------------------------------
--- Distribution for NEWStuck_Clean ---
NEWStuck_Clean
Search (Google)               74491
Community (Stack Overflow)    66410
Context S

In [23]:
# Cleaning the year Columns
experience_cols = ['YearsCode', 'YearsCodePro']
Survey_2021_df_cleaned = clean_years_columns(Survey_2021_df_cleaned, experience_cols)

In [24]:
# Cleaning the Currency column and Salary Columns :  we will extract the currency code from the 'Currency' column, which may contain additional information such as the currency name or symbol. We will then convert the salary columns to numeric values, handling any non-numeric entries appropriately. Finally, we will fill missing values and remove outliers from the salary columns to ensure a more accurate analysis of compensation data.
col = 'Currency'

Survey_2021_df_cleaned[col] = Survey_2021_df_cleaned[col].str.split().str[0]

nan_replacer_cols = ['Currency']
cleaned_nan_cols = ['Currency_Code']

Survey_2021_df_cleaned = normalize_na_replacer_columns(
    Survey_2021_df_cleaned,
    nan_replacer_cols, 
    cleaned_nan_cols,
    replacer_value="Not Available"
)

numeric_target_cols = ['CompTotal', 'ConvertedCompYearly']
Survey_2021_df_cleaned = string_to_numeric(Survey_2021_df_cleaned, numeric_target_cols)

fill_na_and_remove_outlier_percentaile_method(Survey_2021_df_cleaned, 'ConvertedCompYearly', 0.01, 0.95)
fill_na_and_remove_outlier_percentaile_method(Survey_2021_df_cleaned, 'CompTotal', 0.01, 0.95)

,ResponseId,MainBranch,Employment,Country,US_State,UK_Country,EdLevel,Age1stCode,LearnCode,YearsCode,...,Survey_Ease,USA_State,Know_Other_Community,Compensation_Frequency,Gender_Clean,Sexuality_Clean,Ethnicity_Clean,Accessibility_Status,Mental_Health_Status,Currency_Code
0,1,I am a developer by profession,Freelance,Slovakia,NaN,Not Available or Applicable,"Secondary school (e.g. American high school, G...",18 - 24 years,Coding Bootcamp;Other online resources (ex: vi...,0,...,Easy,Not Available or Applicable,No,Monthly,Man,Straight,White,None,None,EUR
1,2,I am a student who is learning to code,Student,Netherlands,NaN,Not Available or Applicable,"BachelorΓÇÖs degree (B.A., B.S., B.Eng., etc.)",11 - 17 years,"Other online resources (ex: videos, blogs, etc...",7,...,Easy,Not Available or Applicable,No,Not Available or Applicable,Man,Straight,White,None,None,Not Available
2,3,"I am not primarily a developer, but I write co...",Student,Russian Federation,NaN,Not Available or Applicable,"BachelorΓÇÖs degree (B.A., B.S., B.Eng., etc.)",11 - 17 years,"Other online resources (ex: videos, blogs, etc...",0,...,Easy,Not Available or Applicable,Yes,Not Available or Applicable,Man,Unknown,Unknown,None,None,Not Available
3,4,I am a developer by profession,Employed,Austria,NaN,Not Available or Applicable,"MasterΓÇÖs degree (M.A., M.S., M.Eng., MBA, etc.)",11 - 17 years,NaN,0,...,Neutral,Not Available or Applicable,No,Monthly,Man,Straight,White,Hearing Impairment,Unknown,EUR
4,5,I am a developer by profession,Freelance,United Kingdom of Great Britain and Northern I...,NaN,England,"MasterΓÇÖs degree (M.A., M.S., M.Eng., MBA, etc.)",5 - 10 years,Friend or family member,17,...,Easy,Not Available or Applicable,No,Not Available or Applicable,Man,Unknown,White,None,Unknown,GBP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83434,83435,I am a developer by profession,Employed,United States of America,Texas,Not Available or Applicable,"BachelorΓÇÖs degree (B.A., B.S., B.Eng., etc.)",11 - 17 years,"Other online resources (ex: videos, blogs, etc...",6,...,Easy,Texas,No,Yearly,Man,Straight,White,None,ADHD/Concentration,USD
83435,83436,I am a developer by profession,Freelance,Benin,NaN,Not Available or Applicable,"BachelorΓÇÖs degree (B.A., B.S., B.Eng., etc.)",11 - 17 years,"Other online resources (ex: videos, blogs, etc...",4,...,Easy,Not Available or Applicable,No,Monthly,Man,Straight,Black,None,None,XOF
83436,83437,I am a developer by profession,Employed,United States of America,New Jersey,Not Available or Applicable,"Secondary school (e.g. American high school, G...",11 - 17 years,School,10,...,Neutral,New Jersey,No,Weekly,Man,Unknown,White,None,None,USD
83437,83438,I am a developer by profession,Employed,Canada,NaN,Not Available or Applicable,"BachelorΓÇÖs degree (B.A., B.S., B.Eng., etc.)",11 - 17 years,Online Courses or Certification;Books / Physic...,5,...,Neutral,Not Available or Applicable,No,Monthly,Man,Straight,White,None,Mood/Emotional,CAD


In [27]:
# 1. Original columns that now have "Clean" versions
raw_redundant_cols = [
    'Employment', 'EdLevel', 'Age1stCode', 'LearnCode', 'OpSys', 
    'OrgSize', 'Trans', 'SOVisitFreq', 'SOAccount', 'SOPartFreq', 
    'SOComm', 'NEWSOSites', 'SurveyLength', 'SurveyEase', 'US_State', 
    'NEWOtherComms', 'CompFreq', 'Currency', 'Gender', 'Sexuality', 
    'Ethnicity', 'Accessibility', 'MentalHealth', 'Age', 'UK_Country'
]

# 2. Raw multi-select strings (already exploded into bridge_results)
multi_select_strings = [
    'DevType', 'NEWStuck', 'LanguageHaveWorkedWith', 'LanguageWantToWorkWith',
    'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'PlatformHaveWorkedWith',
    'PlatformWantToWorkWith', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith',
    'MiscTechHaveWorkedWith', 'MiscTechWantToWorkWith', 'ToolsTechHaveWorkedWith',
    'ToolsTechWantToWorkWith', 'NEWCollabToolsHaveWorkedWith', 'NEWCollabToolsWantToWorkWith'
]

# Combine and drop
total_drop_list = list(set(raw_redundant_cols + multi_select_strings))

Survey_2021_df_cleaned.drop(columns=total_drop_list, inplace=True, errors='ignore')

# Verification
print(f"Final Column Count: {len(Survey_2021_df_cleaned.columns)}")
print(Survey_2021_df_cleaned.columns.tolist())

Final Column Count: 29
['ResponseId', 'MainBranch', 'Country', 'YearsCode', 'YearsCodePro', 'CompTotal', 'ConvertedCompYearly', 'SurveyYear', 'Education_Level', 'AgeCode', 'OperatingSystem', 'Organization_Size', 'TransGender', 'StackOverflow_Visit_Frequency', 'StackOverflow_Account_exists', 'StackOverflow_Participation_Frequency', 'StackOverflow_Community_Experience', 'NewStackOverflow_Sites', 'Survey_Length', 'Survey_Ease', 'USA_State', 'Know_Other_Community', 'Compensation_Frequency', 'Gender_Clean', 'Sexuality_Clean', 'Ethnicity_Clean', 'Accessibility_Status', 'Mental_Health_Status', 'Currency_Code']


In [ ]:
# Writing back to SQL
# Central Fact Table 2021
write_to_sql(df=Survey_2021_df_cleaned,schema='Silver', table_name='Survey_2021', engine=engine)
write_to_sql

# Bridge Tables for Tech Stack and Manual Mapping Columns
for table_name, bridge_df in bridge_results.items():
    write_to_sql(df=bridge_df,schema='Silver', table_name=f"Bridge_{table_name}", engine=engine)

c:\Users\Ayush\Git Repo\Stack-Over-Flow-Survey-Data-Engineering-Project\Data Warehouse\Silver Layer\utils\sql_connector.py:39: FutureWarning: Starting with pandas version 3.0 all arguments of to_sql except for the arguments 'name' and 'con' will be keyword-only.
  df.to_sql(table_name, engine,schema, if_exists='replace', index=False )


DataFrame written to SQL table 'Silver.Survey_2021' successfully.
DataFrame written to SQL table 'Silver.Bridge_DevType_Clean' successfully.
DataFrame written to SQL table 'Silver.Bridge_NEWStuck_Clean' successfully.
